In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))

# ✅ FIX HERE
batch = torch.stack(batch, dim=0)

print(batch.shape)  # should be (2, seq_len)

# =========================
# CONFIG
# =========================

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 264,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# MODEL COMPONENTS
# =========================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        assert (d_out % num_heads == 0)

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5,
            dim=-1
        )

        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)

        context_vec = context_vec.contiguous().view(
            b,
            num_tokens,
            self.d_out
        )

        context_vec = self.out_proj(context_vec)

        return context_vec


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)

        return (
            self.scale * (x - mean) / torch.sqrt(var + self.eps)
            + self.shift
        )


class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (
            1 + torch.tanh(
                torch.sqrt(torch.tensor(2.0 / torch.pi))
                * (x + 0.044715 * x ** 3)
            )
        )


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.att = MultiHeadAttention(
            cfg["emb_dim"],
            cfg["emb_dim"],
            cfg["context_length"],
            cfg["drop_rate"],
            cfg["n_heads"],
            cfg["qkv_bias"]
        )

        self.ff = FeedForward(cfg)

        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])

        self.drop = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        x = x + self.drop(self.att(self.norm1(x)))
        x = x + self.drop(self.ff(self.norm2(x)))
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])

        self.drop = nn.Dropout(cfg["drop_rate"])

        self.blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.norm = LayerNorm(cfg["emb_dim"])

        self.head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, idx):
        B, T = idx.shape

        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))

        x = self.drop(tok + pos)
        x = self.blocks(x)
        x = self.norm(x)

        return self.head(x)

# =========================
# TEXT GENERATION
# =========================

def generate_text_simple(
    model,
    idx,
    max_new_tokens,
    context_size,
    temperature=1.0,
    top_k=50
):

    for _ in range(max_new_tokens):

        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]

        # =============================
        # TEMPERATURE
        # =============================
        logits = logits / temperature

        # =============================
        # TOP-K
        # =============================
        if top_k is not None:

            top_k = min(top_k, logits.size(-1))  # safety

            top_k_values, _ = torch.topk(logits, top_k)

            min_top_k = top_k_values[:, -1].unsqueeze(-1)

            logits = torch.where(
                logits < min_top_k,
                torch.full_like(logits, -torch.inf),
                logits
            )

        # =============================
        # SOFTMAX + SAMPLE
        # =============================
        probas = torch.softmax(logits, dim=-1)

        idx_next = torch.multinomial(probas, num_samples=1)

        idx = torch.cat((idx, idx_next), dim=1)

    return idx

torch.Size([2, 4])


In [2]:
# =========================
# RUN MODEL
# =========================

torch.manual_seed(123)

model = GPTModel(GPT_CONFIG_124M)
model.eval()

batch = batch.to(device)
model.to(device)

out = generate_text_simple(
    model,
    batch,
    max_new_tokens=20,
    context_size=GPT_CONFIG_124M["context_length"],
    temperature=0.8,
    top_k=40
)

print("\nGenerated Token IDs:\n")
print(out)

print("\nDecoded Output:\n")

for row in out:
    print(tokenizer.decode(row.tolist()))

    


Generated Token IDs:

tensor([[ 6109,  3626,  6100,   345,  8026, 39006,  9588, 33218, 48624,  4423,
         37326, 27860, 36057, 32747, 23338, 28904,  8209, 16976, 25449, 21889,
         28768, 37660, 28300, 15736],
        [ 6109,  1110,  6622,   257, 16998, 33194, 39156, 23164, 38691, 15354,
         26487, 25338, 47110, 17988,  3295,  5916, 10739,  4845, 32419, 10384,
          4848, 13423, 21136, 29803]])

Decoded Output:

Every effort moves you Stone psyche salary downstreamnr shutrists citation Utt 305نrily engage specialized transformingottenhamRunning surrogatefloor sensors
Every day holds auraiacistsPred Engineer cardinalWhetherlot fingerprint gestation skeptical Afric fishovie marriageSpecific entrance Fre expects parse Coca


In [3]:
import tensorflow as tf
import tqdm

print("TensorFlow version:", tf.__version__)
print("tqdm version:", tqdm.__version__)

TensorFlow version: 2.21.0
tqdm version: 4.67.3


<div class="alert alert-block alert-success">
    
We download the gpt_download.py Python module directly from this chapter's online repository
</div>

<div class="alert alert-block alert-success">
    
We can now import the download_and_load_gpt2 function from the gpt_download.py
file as follows, which will load the GPT-2 architecture settings (settings) and weight
parameters (params) into our Python session:
</div>

In [4]:
from gpt_download3 import download_and_load_gpt2

In [5]:
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

/opt/anaconda3/envs/torch310/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/checkpoint


/opt/anaconda3/envs/torch310/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/encoder.json


/opt/anaconda3/envs/torch310/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/hparams.json


/opt/anaconda3/envs/torch310/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
model.ckpt.data-00000-of-00001: 100%|████████| 498M/498M [15:35<00:00, 532kiB/s]
/opt/anaconda3/envs/torch310/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
model.ckpt.index: 100%|███████████████████| 5.21k/5.21k [00:00<00:00, 1.48MiB/s]
/opt/anaconda3/envs/torch310/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is 

<div class="alert alert-block alert-success">
    
After the execution of the previous code has been completed, let's inspect the contents of
settings and params:
</div>

In [6]:
print("Settings:", settings)
print("Parameter dictionary keys:", params.keys())

Settings: {'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_layer': 12}
Parameter dictionary keys: dict_keys(['blocks', 'b', 'g', 'wpe', 'wte'])


<div class="alert alert-block alert-success">
    
Both settings and params are Python dictionaries. The settings dictionary stores the LLM
architecture settings similarly to our manually defined GPT_CONFIG_124M settings. 

The
params dictionary contains the actual weight tensors. 

    
Note that we only printed the
dictionary keys because printing the weight contents would take up too much screen space
</div>

<div class="alert alert-block alert-success">
    
We can inspect these weight tensors by printing the whole dictionary via
print(params) or by selecting individual tensors via the respective dictionary keys, for
example, the embedding layer weights:

</div>

In [7]:
print(params["wte"])
print("Token embedding weight tensor dimensions:", params["wte"].shape)

[[-0.11010301 -0.03926672  0.03310751 ... -0.1363697   0.01506208
   0.04531523]
 [ 0.04034033 -0.04861503  0.04624869 ...  0.08605453  0.00253983
   0.04318958]
 [-0.12746179  0.04793796  0.18410145 ...  0.08991534 -0.12972379
  -0.08785918]
 ...
 [-0.04453601 -0.05483596  0.01225674 ...  0.10435229  0.09783269
  -0.06952604]
 [ 0.1860082   0.01665728  0.04611587 ... -0.09625227  0.07847701
  -0.02245961]
 [ 0.05135201 -0.02768905  0.0499369  ...  0.00704835  0.15519823
   0.12067825]]
Token embedding weight tensor dimensions: (50257, 768)


In [8]:
print(params["wte"])
print("Postion embedding weight tensor dimensions:", params["wpe"].shape)

[[-0.11010301 -0.03926672  0.03310751 ... -0.1363697   0.01506208
   0.04531523]
 [ 0.04034033 -0.04861503  0.04624869 ...  0.08605453  0.00253983
   0.04318958]
 [-0.12746179  0.04793796  0.18410145 ...  0.08991534 -0.12972379
  -0.08785918]
 ...
 [-0.04453601 -0.05483596  0.01225674 ...  0.10435229  0.09783269
  -0.06952604]
 [ 0.1860082   0.01665728  0.04611587 ... -0.09625227  0.07847701
  -0.02245961]
 [ 0.05135201 -0.02768905  0.0499369  ...  0.00704835  0.15519823
   0.12067825]]
Postion embedding weight tensor dimensions: (1024, 768)


In [27]:
#print(params["blocks"])

<div class="alert alert-block alert-success">
    
We downloaded and loaded the weights of the smallest GPT-2 model via the
download_and_load_gpt2(model_size="124M", ...) setting. However, note that OpenAI
also shares the weights of larger models: "355M", "774M", and "1558M".

</div>

<div class="alert alert-block alert-success">
    
Above, we loaded the 124M GPT-2 model weights into Python, however we still need to transfer them into our GPTModel instance.

First, we initialize a new GPTModel instance.

Note that the original GPT model initialized the linear layers for the query, key, and value matrices in the multi-head attention module with bias vectors, which is not required or recommended; however, to be able to load the weights correctly, we have to enable these too by setting qkv_bias to True in our implementation, too.
                                                                                                                                                                                                                                                                                                                                  
We are also using the 1024 token context length that was used by the original GPT-2 model(s)

</div>

In [9]:
# Define model configurations in a dictionary for compactness
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# Copy the base configuration and update with specific model settings
model_name = "gpt2-small (124M)"  # Example model name
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])

In [11]:
print("NEW_CONFIG. ",NEW_CONFIG)


NEW_CONFIG.  {'vocab_size': 50257, 'context_length': 1024, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.1, 'qkv_bias': False}


<div class="alert alert-block alert-success">
    
Careful readers may remember that we used a 256-token length earlier, but the original
GPT-2 models from OpenAI were trained with a 1,024-token length, so we have to update
the NEW_CONFIG accordingly:

</div>

<div class="alert alert-block alert-success">
    
Also, OpenAI used bias vectors in the multi-head attention module's linear layers to
implement the query, key, and value matrix computations. 

Bias vectors are not commonly
used in LLMs anymore as they don't improve the modeling performance and are thus
unnecessary. 

However, since we are working with pretrained weights, we need to match the
settings for consistency and enable these bias vectors:

</div>

In [12]:
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})
gpt = GPTModel(NEW_CONFIG)
gpt.eval();

In [13]:
print("NEW_CONFIG. ",NEW_CONFIG)

NEW_CONFIG.  {'vocab_size': 50257, 'context_length': 1024, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.1, 'qkv_bias': True}


<div class="alert alert-block alert-success">
    
By default, the GPTModel instance is initialized with random weights for pretraining. 

The last
step to using OpenAI's model weights is to override these random weights with the weights
we loaded into the params dictionary.

For this, we will first define a small assign utility function that checks whether two
tensors or arrays (left and right) have the same dimensions or shape and returns the
right tensor as trainable PyTorch parameters:
</div>

In [14]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

<div class="alert alert-block alert-success">
    
Next, we define a load_weights_into_gpt function that loads the weights from the params
dictionary into a GPTModel instance gpt:
</div>

<div class="alert alert-block alert-info">

Step 1: Setting the model's positional and token embedding weights to those specified in params.

Step 2: Iterate over each transformer block in the model.

Step 3: The np.split function is used to divide the attention and bias weights into three equal parts for the query,
key, and value components.
    
Step 4: The original GPT-2 model by OpenAI reused the token embedding weights in the output layer to reduce the
total number of parameters, which is a concept known as weight tying.

</div>

In [28]:
import numpy as np

def load_weights_into_gpt(gpt, params):
    
    # mapping postion embedding weight with openAI gpt2 postion embedding weight
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    
    # mapping embedding weight with openAI gpt2 embedding weight
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])

    # params["blocks"] → list of all transformer layers (from pretrained weights)
    # b → current layer index
    # You are loading weights layer by layer
    
    for b in range(len(params["blocks"])):
        # attension block for non qkv_bias
        # in gpt2 created big matrix that is containing query, key,value
        # blocks is dictionary and inside blocks there is another dictionary called transformmer
        # inside of transformmer there is another dictionary called h0,h1,h2,.....h11
        # inside of h0 there is another dictionary called attn
        # inside of attn there is another dictionary called c_attn(feused query key value)
        # inside of c_attn there is another dictionary called w and same like w another is b
        
        q_w, k_w, v_w = np.split((params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        
        # The need for .T comes from a mismatch between:
        # NumPy / OpenAI GPT-2 weights format
        # PyTorch Linear layer format
        # Diff System	                  Weight Shape
        # NumPy (OpenAI GPT-2)	         (input_dim, output_dim)
        # PyTorch nn.Linear	             (output_dim, input_dim)
        
        gpt.blocks[b].att.W_query.weight = assign(gpt.blocks[b].att.W_query.weight, q_w.T)
        gpt.blocks[b].att.W_key.weight = assign(gpt.blocks[b].att.W_key.weight, k_w.T)
        gpt.blocks[b].att.W_value.weight = assign(gpt.blocks[b].att.W_value.weight, v_w.T)

        # attension block for qkv_bias
        # .T is not used for bias because:
        # Bias is a 1D vector, not a 2D matrix — so transpose has no real effect.
        q_b, k_b, v_b = np.split((params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.blocks[b].att.W_query.bias = assign(gpt.blocks[b].att.W_query.bias, q_b)
        gpt.blocks[b].att.W_key.bias = assign(gpt.blocks[b].att.W_key.bias, k_b)
        gpt.blocks[b].att.W_value.bias = assign(gpt.blocks[b].att.W_value.bias, v_b)

        # output projection weight
        gpt.blocks[b].att.out_proj.weight = assign(gpt.blocks[b].att.out_proj.weight, params["blocks"][b]["attn"]["c_proj"]["w"].T)
        # output projection bias
        gpt.blocks[b].att.out_proj.bias = assign(gpt.blocks[b].att.out_proj.bias, params["blocks"][b]["attn"]["c_proj"]["b"])


        ## feed forward neural network for fully connected layer
        # ff.layers[0] → “generate many ideas” (expand)
        # ff.layers[2] → “select useful ideas” (compress)
        # ff.layers[1] is not used because it is not a layer with weights.
        # It is the activation function (GELU).

        # First linear (expand)
        gpt.blocks[b].ff.net[0].weight = assign(
            gpt.blocks[b].ff.net[0].weight,
                params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.blocks[b].ff.net[0].bias = assign(
            gpt.blocks[b].ff.net[0].bias,
                params["blocks"][b]["mlp"]["c_fc"]["b"])

        # Second linear (project)
        gpt.blocks[b].ff.net[2].weight = assign(
            gpt.blocks[b].ff.net[2].weight,
                params["blocks"][b]["mlp"]["c_proj"]["w"].T
            )
        gpt.blocks[b].ff.net[2].bias = assign(
            gpt.blocks[b].ff.net[2].bias,
                params["blocks"][b]["mlp"]["c_proj"]["b"]
                )
        # normalization 1
        gpt.blocks[b].norm1.scale = assign(gpt.blocks[b].norm1.scale, params["blocks"][b]["ln_1"]["g"])
        gpt.blocks[b].norm1.shift = assign(gpt.blocks[b].norm1.shift, params["blocks"][b]["ln_1"]["b"])

        # normalization 2
        gpt.blocks[b].norm2.scale = assign(gpt.blocks[b].norm2.scale, params["blocks"][b]["ln_2"]["g"])
        gpt.blocks[b].norm2.shift = assign(gpt.blocks[b].norm2.shift, params["blocks"][b]["ln_2"]["b"])

    # final normalization
    gpt.norm.scale = assign(gpt.norm.scale, params["g"])
    gpt.norm.shift = assign(gpt.norm.shift, params["b"])
    gpt.head.weight = assign(gpt.head.weight, params["wte"])



<div class="alert alert-block alert-success">

In the load_weights_into_gpt function, we carefully match the weights from OpenAI's
implementation with our GPTModel implementation. 

To pick a specific example, OpenAI
stored the weight tensor for the output projection layer for the first transformer block as
params["blocks"][0]["attn"]["c_proj"]["w"]. 
                                                        
In our implementation, this weight
tensor corresponds to gpt.trf_blocks[b].att.out_proj.weight, where gpt is a
GPTModel instance.
</div>

<div class="alert alert-block alert-success">

Developing the load_weights_into_gpt function took a lot of guesswork since OpenAI
used a slightly different naming convention from ours. 

However, the assign function would
alert us if we try to match two tensors with different dimensions. 

Also, if we made a
mistake in this function, we would notice this as the resulting GPT model would be unable
to produce coherent text.
    
</div>

<div class="alert alert-block alert-success">

Let's now try the load_weights_into_gpt out in practice and load the OpenAI model
weights into our GPTModel instance gpt:
    
</div>

In [29]:
load_weights_into_gpt(gpt, params)
gpt.to(device);

In [35]:

torch.manual_seed(123)

model = GPTModel(GPT_CONFIG_124M)
model.eval()

batch = batch.to(device)
model.to(device)

out = generate_text_simple(
    model,
    batch,
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"],
    temperature=1.4,
    top_k=100
)

print("\nGenerated Token IDs:\n")
print(out)

print("\nDecoded Output:\n")

for row in out:
    print(tokenizer.decode(row.tolist()))



Generated Token IDs:

tensor([[ 6109,  3626,  6100,   345, 40145, 23447, 18676, 12394, 28106, 39181,
         20603, 35563, 47885, 24600,  5120, 28904, 45654, 41780, 29676, 49042,
         41695,  1209, 27567, 29280, 15465, 27455, 30168, 29951, 24184],
        [ 6109,  1110,  6622,   257, 34117, 39870, 26139, 29669, 38797, 39889,
         43583, 25233,  9249, 47664,  6613, 44104,  4782, 39324, 31544, 22487,
         48236, 26180, 10982, 12060, 28392, 36957, 23734, 25695, 13901]])

Decoded Output:

Every effort moves you JPM WishcastleINT HonestStretch cad roam rake Filip proceedrily dopeusional anarchists Atkinson NRL�ibi pervasive Shardar misguided notwithstanding customize
Every day holds a encryptgm Millennium empowered ferment chuckled Starfleet Chevitches Orth proud (_uctKenn PersebottomToyν Fantasy Lie Jaguars Stress380camera electoral
